## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array

: 

## Parameters

In [ ]:
img_width = 160
img_height = 40

## Label Encoding Setup

In [ ]:
# Character map: 0–9 + CTC blank
characters = list("0123456789")
# char to int
char_to_num = tf.keras.layers.StringLookup(
    vocabulary=characters,
    mask_token=None,
    num_oov_indices=0
)

vocab = char_to_num.get_vocabulary()
print(vocab)
# int to char
num_to_char = tf.keras.layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(),
    mask_token=None,
    num_oov_indices=0,
    invert=True
)

## Load Labels CSV

In [ ]:
labels_df = pd.read_csv("./DIDA_12000_String_Digit_Labels.xls")

# Ensure columns: filename and label
labels_df.columns = ['filename', 'label']

# Shuffle
labels_df = labels_df.sample(frac=1).reset_index(drop=True)

## Data Generator with tf.data

In [ ]:
# def preprocess_image(filename):
#     img = load_img(filename, color_mode='grayscale', target_size=(img_height, img_width))
#     img = img_to_array(img) / 255.0
#     return img

# def encode_label(label):
#     label = tf.strings.unicode_split(label, input_encoding='UTF-8')
#     label = char_to_num(label)
#     return label

# def load_data(row):
#     img_path = os.path.join("DIDA_1", row['filename'])
#     img = preprocess_image(img_path)
#     label = encode_label(row['label'])
#     return img, label

# Convert dataframe to dataset
# def create_dataset(df):
#     img_paths = df['filename'].values
#     labels = df['label'].values
#     dataset = tf.data.Dataset.from_tensor_slices((img_paths, labels))

#     # def map_fn(fname, label):
#     #     img = preprocess_image(os.path.join("DIDA_1", fname.numpy().decode()))
#     #     lbl = encode_label(label)
#     #     return img, lbl

#     def map_fn(fname, label):
#         fname_str = fname.numpy().decode('utf-8') if isinstance(fname.numpy(), bytes) else str(fname.numpy())
#         img = preprocess_image(os.path.join("DIDA_1", fname_str))
#         lbl = encode_label(label)
#         return img, lbl

#     dataset = dataset.map(
#         lambda fname, label: tf.py_function(map_fn, [fname, label], [tf.float32, tf.int64]),
#         num_parallel_calls=tf.data.AUTOTUNE
#     )

#     padding_values = (0.0, tf.constant(-1, dtype=tf.int64))  # match label type

#     dataset = dataset.padded_batch(
#         batch_size,
#         padded_shapes=([img_height, img_width, 1], [max_label_len]),
#         padding_values=padding_values,
#         drop_remainder=True
#     )

#     dataset = dataset.prefetch(tf.data.AUTOTUNE)
#     return dataset


labels_df['label'] = labels_df['label'].astype(str)
# Convert dataframe to dataset
def create_dataset(df, batch_size = 32):
    img_paths = df['filename'].apply(lambda x: os.path.join("DIDA_1", str(x)) + ".jpg").values
    labels = df['label'].values
    # create tuple (path, label)
    dataset = tf.data.Dataset.from_tensor_slices((img_paths, labels))

    def map_fn(fname, label):
        img = tf.io.read_file(fname)
        img = tf.io.decode_jpeg(img, channels=1) # Decodierung zu Tensor der Form [H, W, 1] (Graustufen)
        img = tf.image.resize(img, [img_height, img_width]) # auf einheitliche Größe resized
        img = tf.cast(img, tf.float32) / 255.0 # Normalisierung

        label = tf.strings.unicode_split(label, input_encoding='UTF-8') # "1836" -> ['1', '8', '3', '6']
        label = char_to_num(label) # ['1', '8', '3', '6'] -> [1, 8, 3, 6]
        return img, label

    dataset = dataset.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)

    # dynamically pad to the longest sequence in each batch
    dataset = dataset.padded_batch(
        batch_size,
        # tensorflow: pad label sequences to the longest in the current batch
        padded_shapes=([img_height, img_width, 1], [None]), 
        padding_values=(0.0, tf.constant(-1, dtype=tf.int64)),
        drop_remainder=True
    ) # [1, 2, -1, -1] with -1 as padding

    return dataset.prefetch(tf.data.AUTOTUNE)

# Split train/val
train_size = int(0.9 * len(labels_df))
train_df = labels_df[:train_size]
val_df = labels_df[train_size:]

train_ds = create_dataset(train_df)
val_ds = create_dataset(val_df)

## Define CNN + BiLSTM Model

In [ ]:
## grayscale (height, width, channel)
input_img = layers.Input(shape=(img_height, img_width, 1), name='image')
## for CTC loss later (placeholder for now)
labels = layers.Input(name='label', shape=(None,), dtype='int64')

# CNN
## 32 filters using 3x3 kernel
x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(input_img)
## downsampling by half
x = layers.MaxPooling2D((2,2))(x)
## for more complex patterns: 64 filters
x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
## downsampling by half
x = layers.MaxPooling2D((2,2))(x)

# Reshape 2D -> 1D
new_shape = (img_width // 4, (img_height // 4) * 64)
x = layers.Reshape(target_shape=new_shape)(x)

# BiLSTM (recurrent layer that processes sequence forward and backwards)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
## output layer
x = layers.Dense(len(characters) + 1, activation='softmax')(x) # +1 notwendig für CTC 

model = tf.keras.Model(inputs=input_img, outputs=x)

## Define CTC Loss Model

In [ ]:
# def ctc_loss_lambda_func(args):
#     y_pred, labels = args
#     batch_size = tf.shape(y_pred)[0]
#     input_len = tf.ones(shape=(batch_size, 1), dtype='int32') * tf.shape(y_pred)[1]
#     label_len = tf.math.count_nonzero(labels, axis=-1, keepdims=True)
#     loss = tf.keras.backend.ctc_batch_cost(labels, y_pred, input_len, label_len)
#     return loss

def ctc_loss_lambda_func(args):
    y_pred, labels = args

    # Compute actual lengths (ignoring padding -1)
    label_len = tf.reduce_sum(tf.cast(tf.not_equal(labels, -1), dtype=tf.int32), axis=-1, keepdims=True)

    # Time steps of output (for ctc_batch_cost)
    input_len = tf.ones(shape=(tf.shape(y_pred)[0], 1), dtype=tf.int32) * tf.shape(y_pred)[1]

    loss = tf.keras.backend.ctc_batch_cost(labels, y_pred, input_len, label_len)
    return loss

# loss_out = layers.Lambda(
#     ctc_loss_lambda_func,
#     output_shape=(1,),  # Important for compatibility
#     name="ctc_loss"
# )([model.output, labels])
loss_output = layers.Lambda(ctc_loss_lambda_func)([model.output, labels])

# Training model
ctc_model = tf.keras.Model(inputs=[input_img, labels], outputs=loss_output)
ctc_model.compile(optimizer='adam', loss=lambda y_true, y_pred: y_pred)  # DO NOT provide loss here!

# output = layers.Lambda(
#     ctc_loss_lambda_func,
#     output_shape=(1,),  # Shape of loss per sample
#     name="ctc_loss"
# )([model.output, labels])
# ctc_model = tf.keras.models.Model(inputs=[input_img, labels], outputs=output)
# ctc_model.compile(optimizer='adam')

## Training

In [ ]:
# Dummy loss formatting
def format_batch(imgs, lbls):
    batch_len = tf.shape(imgs)[0]
    dummy_loss = tf.zeros((batch_len, 1))  # shape (batch_size, 1)
    return {'image': imgs, 'label': lbls}, dummy_loss

history = ctc_model.fit(
    train_ds.map(format_batch),
    validation_data=val_ds.map(format_batch), # 2 inputs mappen
    epochs=20
)

# train_ds_mapped = train_ds.map(format_batch)
# val_ds_mapped = val_ds.map(format_batch)
# history = ctc_model.fit(
#     train_ds_mapped,
#     validation_data=val_ds_mapped,
#     epochs=20
# )

## Inference (decode predictions)

In [ ]:
def decode_predictions(pred):
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    results = tf.keras.backend.ctc_decode(pred, input_length=input_len, greedy=True)[0][0]
    output_text = []
    # Sequence of indices to text = "1836"
    for res in results:
        res = tf.gather(res, tf.where(res != -1))[:, 0]
        output_text.append(tf.strings.reduce_join(num_to_char(res)).numpy().decode())
    return output_text

# Predict batch
batch_imgs, _ = next(iter(val_ds))
preds = model.predict(batch_imgs)
texts = decode_predictions(preds)
print(texts[:10])

## Accuracy Evaluation

In [ ]:
correct = 0
total = 0

for batch in val_ds.take(20):
    imgs, lbls = batch
    preds = model.predict(imgs)
    decoded = decode_predictions(preds)

    for i in range(len(decoded)):
        true = tf.strings.reduce_join(num_to_char(lbls[i][lbls[i] != -1])).numpy().decode()
        pred = decoded[i]
        if true == pred:
            correct += 1
        total += 1

print(f"Validation Accuracy: {correct / total:.2%}")